# Lab | EDA Bivariate Analysis: Amazon UK Product Insights Part II

This notebook covers best-seller trends, prices, ratings, outlier removal, statistical tests, and price-rating relationships.

## Load the dataset

Download the Kaggle CSV and save it as amazon_uk_products.csv in the same folder as this notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

candidate_paths = [Path("amazon_uk_products.csv"), Path("amazon_products.csv"), Path("amazon.csv")]
existing_paths = [path for path in candidate_paths if path.exists()]
if not existing_paths:
    raise FileNotFoundError("Download the Kaggle CSV and save it as amazon_uk_products.csv in the same folder as this notebook.")

df = pd.read_csv(existing_paths[0])
df.columns = (df.columns.astype(str).str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
df = df.rename(columns={"categoryname": "category", "category_name": "category", "isbest_seller": "is_best_seller", "isbestseller": "is_best_seller", "rating": "stars"})
required = {"category", "price", "stars", "is_best_seller"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
df["price"] = pd.to_numeric(df["price"].astype(str).str.replace("£", "", regex=False).str.replace(",", "", regex=False), errors="coerce")
df["stars"] = pd.to_numeric(df["stars"], errors="coerce")
if df["is_best_seller"].dtype != bool:
    df["is_best_seller"] = df["is_best_seller"].astype(str).str.strip().str.lower().map({"true": True, "1": True, "yes": True, "false": False, "0": False, "no": False})
df = df.dropna(subset=["category", "price", "stars", "is_best_seller"]).copy()
print("Dataset shape after basic cleaning:", df.shape)
print(df[["category", "price", "stars", "is_best_seller"]].head())

## Part 1: Best-seller trends across categories

In [ ]:
best_seller_crosstab = pd.crosstab(df["category"], df["is_best_seller"]).reindex(columns=[False, True], fill_value=0)
best_seller_counts = best_seller_crosstab.reindex(columns=[False, True], fill_value=0)
best_seller_crosstab["best_seller_rate"] = best_seller_counts[True] / best_seller_counts.sum(axis=1)
best_seller_crosstab = best_seller_crosstab.sort_values("best_seller_rate", ascending=False)
print(best_seller_crosstab)

contingency_table = best_seller_crosstab.reindex(columns=[False, True], fill_value=0)
chi2_statistic, chi2_p_value, degrees_of_freedom, expected = stats.chi2_contingency(contingency_table)
sample_size = contingency_table.to_numpy().sum()
cramers_v = (chi2_statistic / (sample_size * min(contingency_table.shape[0] - 1, contingency_table.shape[1] - 1))) ** 0.5
print(f"\nChi-square statistic: {chi2_statistic:.3f}")
print(f"p-value: {chi2_p_value:.6g}")
print(f"Degrees of freedom: {degrees_of_freedom}")
print(f"Cramer's V: {cramers_v:.3f}")
print("Association conclusion:", "significant at alpha = 0.05" if chi2_p_value < 0.05 else "not significant at alpha = 0.05")

plot_counts = pd.crosstab(df["category"], df["is_best_seller"]).reindex(columns=[False, True], fill_value=0)
plot_counts.columns = ["Not best-seller", "Best-seller"]
plot_counts = plot_counts.loc[plot_counts.sum(axis=1).sort_values(ascending=False).index]
plot_counts.plot(kind="bar", stacked=True, figsize=(14, 6), color=["#b7c9e2", "#e76f51"])
plt.title("Best-seller status by product category")
plt.xlabel("Category")
plt.ylabel("Number of products")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

The chi-square p-value indicates whether best-seller status depends on category. Cramér's V indicates the association strength: values close to 0 are weak, while larger values indicate a stronger relationship.

## Part 2: Prices and ratings across categories

In [ ]:
price_q1, price_q3 = df["price"].quantile([0.25, 0.75])
price_iqr = price_q3 - price_q1
price_lower_bound = price_q1 - 1.5 * price_iqr
price_upper_bound = price_q3 + 1.5 * price_iqr
df_no_price_outliers = df[df["price"].between(price_lower_bound, price_upper_bound)].copy()
print(f"IQR price bounds: {price_lower_bound:.2f} to {price_upper_bound:.2f}")
print("Rows removed:", len(df) - len(df_no_price_outliers))

# Violin plot for the 20 largest categories.
top_20 = df_no_price_outliers["category"].value_counts().head(20).index
violin_data = df_no_price_outliers[df_no_price_outliers["category"].isin(top_20)]
plt.figure(figsize=(16, 7))
sns.violinplot(data=violin_data, x="category", y="price", inner="quartile", cut=0)
plt.title("Price distribution for the 20 largest categories")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()
median_price_by_category = df_no_price_outliers.groupby("category")["price"].median()
print("Highest median price category:", median_price_by_category.idxmax(), median_price_by_category.max())

# Average price for the 10 largest categories, plus the all-category maximum.
top_10 = df_no_price_outliers["category"].value_counts().head(10).index
average_price_top_10 = df_no_price_outliers[df_no_price_outliers["category"].isin(top_10)].groupby("category")["price"].mean().sort_values(ascending=False)
average_price_top_10.plot(kind="bar", figsize=(12, 5), color="#457b9d")
plt.title("Average price for the 10 largest categories")
plt.ylabel("Average price")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()
average_price_by_category = df_no_price_outliers.groupby("category")["price"].mean()
print("Highest average price category:", average_price_by_category.idxmax(), average_price_by_category.max())

# Rating box plot for the 10 largest categories and median across all categories.
rating_data = df_no_price_outliers[df_no_price_outliers["category"].isin(top_10)]
plt.figure(figsize=(12, 5))
sns.boxplot(data=rating_data, x="category", y="stars")
plt.title("Rating distribution for the 10 largest categories")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()
median_rating_by_category = df_no_price_outliers.groupby("category")["stars"].median()
print("Highest median rating category:", median_rating_by_category.idxmax(), median_rating_by_category.max())

## Part 3: Price and rating relationship

In [ ]:
correlation_data = df_no_price_outliers[["price", "stars"]].dropna()
pearson_r, pearson_p_value = stats.pearsonr(correlation_data["price"], correlation_data["stars"])
print(f"Pearson correlation: {pearson_r:.4f}")
print(f"p-value: {pearson_p_value:.6g}")
print("Correlation conclusion:", "statistically significant" if pearson_p_value < 0.05 else "not statistically significant")

scatter_data = correlation_data.sample(min(len(correlation_data), 5000), random_state=42)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=scatter_data, x="price", y="stars", alpha=0.35, s=25)
sns.regplot(data=scatter_data, x="price", y="stars", scatter=False, color="red")
plt.title("Product price versus rating")
plt.tight_layout()
plt.show()

numeric_columns = df_no_price_outliers.select_dtypes(include="number").columns
plt.figure(figsize=(10, 7))
sns.heatmap(df_no_price_outliers[numeric_columns].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap of numerical variables")
plt.tight_layout()
plt.show()

stats.probplot(df_no_price_outliers["price"].dropna(), dist="norm", plot=plt)
plt.title("Q-Q plot of product prices")
plt.tight_layout()
plt.show()
print("A curved Q-Q pattern indicates that prices are not normally distributed; a roughly straight pattern indicates approximate normality.")